## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 4a: Regresie: de la medie la model</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/04-regresie/curs-04a-regresie-uni-multivariata.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "04-regresie"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Cum lucrăm azi: tu, agentul și notebook-ul

Acest curs se joacă în trei: **lecția** (acest notebook), **tu** și **agentul tău** (Gemini în Colab, un agent din VS Code sau din Antigravity). Agentul scrie codul; tu decizi ce se cere, de ce, și verifici dacă rezultatul are sens. Fiecare idee este un **capitol** cu aceiași pași:

| Pas | Ce faci |
|---|---|
| **Concept** | citești 2-3 propoziții |
| **Prezice** | spui (sau scrii) ce crezi că va ieși, *înainte* să rulezi |
| **Cere agentului** | formulezi cererea; apoi rulezi codul de referință și îl compari cu ce a scris agentul |
| **Verifică** | compari cu valorile de control |
| **Capcană** | vezi ce poate merge prost, chiar dacă codul rulează |
| **Pentru client** | spui rezultatul în limbaj de manager |

**Primer** (lipește-l agentului la începutul conversației; dacă ai pierdut conversația sau agentul a rămas fără tokeni, lipește primerul și *Jurnalul de sesiune* de la finalul notebook-ului unui alt agent și scrie „continuă de aici”):

```text
Rolul tău: ești partenerul meu de lucru la cursul SEED1 (analiză de date și învățare automată), modulul 4: regresie (regresie liniară simplă și multiplă, supraînvățare, regularizare Ridge, Lasso, Elastic Net). Eu sunt student și nu știu să programez în Python; eu decid ce se face și de ce, tu scrii codul, iar eu verific rezultatul.

Datele cursului:
- Se încarcă DOAR cu `from seed.date import incarca`. `incarca("lab-set6")` întoarce un dicționar nume_tabel -> pandas DataFrame; tabelul cu prețuri se ia cu `incarca("lab-set6", "train")`. Tabelul `test` NU are coloana țintă `SalePrice`.
- Nu folosi căi de fișiere și nu descărca date de pe internet.
- Folosește pandas, matplotlib/seaborn și scikit-learn. Cod scurt, comentarii în română, un pas pe rând.

Reguli pentru acest modul (le aplici chiar dacă nu le cer eu):
1. Scoate ID-urile (`Id`) și ținta din predictori. Împarte train/test (20% test, `random_state=42`) ÎNAINTE de orice transformare învățată din date (completarea lipsurilor, one-hot, scalare).
2. Pune pregătirea și modelul într-un `Pipeline`, ca transformările să fie învățate doar pe antrenare.
3. Compară întotdeauna cu modelul de referință (media) și raportează MAE, RMSE și R² pe antrenare ȘI pe test.
4. Hiperparametrii (de exemplu `alpha`) se aleg prin validare încrucișată pe antrenare, niciodată după scorul pe test.
5. Nu inventa coloane: dacă nu ești sigur, afișează `df.columns`.
6. Spune-mi dacă ceva din rezultat îți pare ciudat (diferență mare între antrenare și test, coeficienți cu semn absurd), în loc să-l prezinți ca pe un succes.

Cum lucrăm:
- Implicit ești în MOD TUTORE: înainte de a-mi da codul, îmi pui o singură întrebare scurtă despre concept („ce crezi că se va întâmpla?”). Nu rezolvi temele în locul meu; mă ajuți să verific.
- La cererea mea poți trece în MOD EXECUTANT (execuți direct și spui pe scurt ce ai făcut) sau în MOD RECENZENT (cauți greșeli în ce îți arăt: ID-uri printre predictori, evaluare pe datele de antrenare, transformări învățate înainte de împărțire).
- Când scriu „închide capitolul”, completezi Jurnalul de sesiune în formatul de mai jos, numai din ce am discutat. Dacă ceva nu s-a discutat, scrii „nu știu”.

Formatul Jurnalului de sesiune:
## Jurnal de sesiune
- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):

Răspunde acum doar cu: rolul tău în două propoziții și cele 6 reguli, pe scurt.
```

## Problema de azi

O agenție imobiliară din Ames (Iowa): *„Cât să cerem pe o casă și cât de mult poate greși estimarea voastră?”*

Avem 1.460 de case vândute, cu prețul (`SalePrice`) și 79 de caracteristici. Spre deosebire de modulul 3, acum **avem o țintă**: un număr pe care modelul învață să-l prezică din celelalte coloane. Aceasta este **regresia** (învățare supervizată cu țintă numerică).

În modulul 5 ținta va fi o categorie (pleacă / rămâne): **clasificare**. Proiectul vostru este un clasificator, dar tot ce construim azi (împărțirea, modelul de referință, supraînvățarea, pregătirea în `Pipeline`, regularizarea) se folosește acolo neschimbat.

**Planul de azi (≈55 min):** capitolul 0 cererea vagă · 1 împărțirea și ghicitul mediei · 2 o dreaptă · 3 mai mulți predictori · 4 prea multe coloane, prea puține case. Regularizarea și ce îi spunem clientului: `curs-04b`. Definițiile complete: secțiunea **Referință** de la sfârșit.

## Capitolul 0. Cererea vagă

**Concept.** Un set de date dintr-o competiție Kaggle vine de obicei în două tabele: `train` (cu răspunsul, adică prețul) și `test` (fără răspuns: e ce trebuia prezis în competiție). Un model nu poate fi evaluat pe un tabel fără răspunsuri.

**Prezice.** Cerem agentului: *„prezice prețul caselor din `lab-set6`”*. Ce tabel crezi că va folosi? Cum își va măsura reușita?

**Cere agentului** (mod executant):

> Mod executant: încarcă `lab-set6` și spune-mi ce tabele are, câte rânduri și coloane are fiecare și în care dintre ele există coloana `SalePrice`.

In [ ]:
# Cod de referință: ce tabele avem și unde este ținta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from seed.date import incarca

date = incarca("lab-set6")
for nume, tabel in date.items():
    print(f"{nume:18s} {str(tabel.shape):12s} are SalePrice: {'SalePrice' in tabel.columns}")

df = date["train"]
df[["Id", "GrLivArea", "OverallQual", "YearBuilt", "Neighborhood", "SalePrice"]].head()

**Verifică.** `train` (1460, 81) conține `SalePrice`; `test` (1459, 80) nu o conține; `sample_submission` (1459, 2) are o coloană `SalePrice`, dar cu **prețuri de exemplu** (formatul unui răspuns la competiție), nu cu prețuri reale.

**Capcană.** Un agent grăbit antrenează pe `train` și apoi „evaluează” pe `test` lipit de `sample_submission`, ca și cum acelea ar fi prețuri adevărate. Sau, mai rău, raportează eroarea pe aceleași case pe care a învățat (capitolul 1). O coloană care *se numește* `SalePrice` nu e neapărat prețul real. Pe casele din `test` nu știm prețurile, deci nu putem afla cât greșim. **Toată evaluarea de azi se face pe o parte din `train` pusă deoparte.**

**Pentru client.** „Lucrăm cu 1.460 de vânzări cu preț cunoscut; din ele păstrăm o parte pe care modelul nu o vede, ca să măsurăm cinstit cât greșește.”

## Capitolul 1. Împărțirea și ghicitul mediei

**Concept.** Un model se judecă pe case **pe care nu le-a văzut**. Împărțim datele o singură dată, la început: 80% pentru **antrenare**, 20% pentru **test**. Tot ce se învață din date (completarea lipsurilor, codificarea, scalarea, alegerea parametrilor) se învață doar din antrenare.

Primul „model” este și cel mai simplu: **ghicim media** prețurilor din antrenare pentru fiecare casă. Orice model adevărat trebuie să-l bată. Îl măsurăm cu trei numere ($y$ = prețul real, $\hat y$ = prețul estimat):

$$\text{MAE} = \frac{1}{n}\sum_i |y_i - \hat y_i| \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i - \hat y_i)^2} \qquad R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$$

MAE și RMSE sunt în **dolari** (RMSE pedepsește mai tare greșelile mari). $R^2$ spune ce parte din variația prețurilor explică modelul: 1 = perfect, 0 = cât media.

**Prezice.** Dacă ghicim media pentru fiecare casă, cu cât greșim în medie: 10.000 $? 60.000 $? Cât e $R^2$?

**Cere agentului:**

> Împarte tabelul `train` din `lab-set6` în 80% antrenare și 20% test (`random_state=42`), cu ținta `SalePrice` și fără coloana `Id` printre predictori. Prezice pentru fiecare casă din test media prețurilor din antrenare și raportează MAE, RMSE și R².

In [ ]:
# Cod de referință: împărțirea (o singură dată) și modelul de referință
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df.drop(columns=["Id", "SalePrice"])   # predictorii: fără ID și fără țintă
y = df["SalePrice"]                        # ținta
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Antrenare:", X_train.shape, "  Test:", X_test.shape)

def metrici(y_adevarat, y_prezis):
    """MAE și RMSE (în dolari) și R²."""
    return {"MAE": round(mean_absolute_error(y_adevarat, y_prezis)),
            "RMSE": round(np.sqrt(mean_squared_error(y_adevarat, y_prezis))),
            "R2": round(r2_score(y_adevarat, y_prezis), 3)}

media = y_train.mean()
print("Media prețurilor în antrenare:", round(media))
print("Modelul de referință, pe test:", metrici(y_test, np.full(len(y_test), media)))

**Verifică.** Antrenare (1168, 79), test (292, 79); media 181.442 $; pe test MAE ≈ 62.576 $, RMSE ≈ 87.619 $, $R^2$ ≈ −0,001.

**Capcană.** Un $R^2$ de 0,8 sună bine doar dacă știm de unde pornim. Pe test, media dă $R^2 \approx 0$ (poate ieși chiar ușor negativ, pentru că media e calculată pe antrenare, nu pe test). Dacă agentul raportează un scor fără să spună pe ce date, întreabă: *antrenare sau test?*

**Pentru client.** „Fără niciun model, o estimare «după medie» greșește în medie cu ~62.600 $ pe casă. Asta e ștacheta pe care orice model trebuie s-o treacă.”

## Capitolul 2. O dreaptă

**Concept.** Regresia liniară simplă caută dreapta $\hat y = b_0 + b_1 x$ care trece „cel mai aproape” de puncte: cea cu cea mai mică sumă a pătratelor erorilor (**metoda celor mai mici pătrate**, Referința R2). **Panta** $b_1$ spune cu câți dolari crește prețul estimat când $x$ crește cu o unitate. **Interceptul** $b_0$ este prețul estimat pentru $x = 0$, adesea fără sens practic. **Reziduul** $y - \hat y$ este cât a greșit modelul la o casă.

Predictorul: `GrLivArea`, suprafața locuibilă în **picioare pătrate** (1 sqft ≈ 0,093 m²).

**Prezice.** Cu câți dolari crește prețul pentru fiecare sqft în plus? Cu cât va bate dreapta modelul-medie?

**Cere agentului:**

> Pe aceeași împărțire, antrenează o regresie liniară `SalePrice` ~ `GrLivArea`. Afișează panta și interceptul, MAE/RMSE/R² pe antrenare și pe test și un grafic de împrăștiere cu dreapta. Marchează casele cu peste 4.000 sqft.

In [ ]:
# Cod de referință: regresia liniară simplă
from sklearn.linear_model import LinearRegression

dreapta = LinearRegression().fit(X_train[["GrLivArea"]], y_train)
print(f"Panta b1 = {dreapta.coef_[0]:.1f} $ pe sqft    Interceptul b0 = {dreapta.intercept_:,.0f} $")
print("Antrenare:", metrici(y_train, dreapta.predict(X_train[["GrLivArea"]])))
print("Test:     ", metrici(y_test, dreapta.predict(X_test[["GrLivArea"]])))

mari = df[df["GrLivArea"] > 4000]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X_train["GrLivArea"], y_train, s=8, alpha=0.4, label="case (antrenare)")
xs = pd.DataFrame({"GrLivArea": np.linspace(0, X["GrLivArea"].max(), 100)})
ax.plot(xs["GrLivArea"], dreapta.predict(xs), color="C3", label="dreapta de regresie")
ax.scatter(mari["GrLivArea"], mari["SalePrice"], s=80, facecolors="none", edgecolors="k", label="peste 4.000 sqft")
ax.set_xlabel("GrLivArea (sqft)")
ax.set_ylabel("SalePrice ($)")
ax.set_title("Prețul crește cu suprafața, dar nu pentru toate casele mari")
ax.legend()
plt.tight_layout()
plt.show()
mari[["Id", "GrLivArea", "SalePrice", "SaleCondition"]]

**Verifică.** Panta ≈ 102,5 $ pe sqft (≈ 1.100 $ pe m²), interceptul ≈ 24.900 $; pe test MAE ≈ 38.341 $ și $R^2$ ≈ 0,554 (pe antrenare 0,484). Patru case au peste 4.000 sqft.

**Capcană.** Două dintre casele mari (Id 524 și 1299) s-au vândut cu mai puțin de 185.000 $, ca vânzări „parțiale” (`SaleCondition = Partial`: case încă neterminate). Câteva puncte atipice pot înclina toată dreapta. Agentul nu le semnalează dacă nu-i ceri; **uită-te la grafic înainte de a crede panta**.

**Pentru client.** „Fiecare sqft în plus adaugă în medie ~100 $ la preț (~1.100 $ pe m²). Doar din suprafață greșim în medie cu ~38.000 $: mult mai bine decât ~62.600 $ fără model, dar încă mult.”

## Capitolul 3. Mai mulți predictori

**Concept.** Regresia multiplă adaugă predictori: $\hat y = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_p x_p$. Interpretarea unui coeficient se schimbă: $b_j$ este efectul lui $x_j$ **când toți ceilalți predictori rămân la fel**. De aceea coeficientul unei coloane depinde de ce alte coloane sunt în model: două coloane care spun aproape același lucru (corelate) își împart efectul.

Coloanele text (cartierul, tipul de acoperiș) intră în model după **codificarea one-hot** (modulul 3, `curs-03a`, Referința R1b); o facem în capitolul 4.

**Prezice.** Singur, `GarageCars` (câte mașini încap în garaj) „valorează” mult pe loc de parcare. Ce se întâmplă cu acest coeficient dacă adăugăm suprafața garajului, calitatea, suprafața casei și a subsolului și anul construcției?

**Cere agentului:**

> Pe antrenare, antrenează două regresii: `SalePrice` ~ `GarageCars` și `SalePrice` ~ `GarageCars` + `GarageArea` + `GrLivArea` + `OverallQual` + `TotalBsmtSF` + `YearBuilt`. Pune coeficienții într-un tabel, afișează corelația dintre `GarageCars` și `GarageArea` și raportează MAE și R² pe test pentru al doilea model.

In [ ]:
# Cod de referință: același predictor, singur și împreună cu alții
sase = ["GarageCars", "GarageArea", "GrLivArea", "OverallQual", "TotalBsmtSF", "YearBuilt"]
singur = LinearRegression().fit(X_train[["GarageCars"]], y_train)
multiplu = LinearRegression().fit(X_train[sase], y_train)

coeficienti = pd.DataFrame({"singur": pd.Series(singur.coef_, index=["GarageCars"]),
                            "cu ceilalți 5": pd.Series(multiplu.coef_, index=sase)}).round(1)
print(coeficienti)
print("\nCorelația GarageCars - GarageArea:", round(X_train["GarageCars"].corr(X_train["GarageArea"]), 2))
print("6 predictori, test:", metrici(y_test, multiplu.predict(X_test[sase])))

**Verifică.** `GarageCars` singur ≈ 66.911 $ pe loc de parcare; cu ceilalți 5 predictori ≈ 11.202 $. Corelația cu `GarageArea` ≈ 0,88. Modelul cu 6 predictori: MAE pe test ≈ 25.172 $, $R^2$ ≈ 0,795.

**Capcană.** Singur, `GarageCars` „lua asupra lui” tot ce vine de obicei cu un garaj mare: case mai noi, mai mari, mai bune. Cu acestea în model, rămâne doar efectul locului de parcare în sine. **Un coeficient nu se citește izolat.** Încă două capcane din acest set:
- `MSSubClass` (tipul clădirii) are cifre (20, 60, 190), dar e o **categorie**, nu o măsură: 60 nu e „de trei ori” 20;
- la `PoolQC`, `Alley`, `Fence`, lipsa valorii înseamnă „nu are”, nu „nu știm”.

**Pentru client.** „Cu șase caracteristici greșim în medie cu ~25.000 $. Prețul vine mai ales din calitatea generală și din suprafață; garajul contează, dar mai puțin decât pare la prima vedere.”

## Capitolul 4. Prea multe coloane, prea puține case

**Concept.** Un model cu multe coloane poate **memora** casele din antrenare în loc să învețe reguli generale: **supraînvățare** (*overfitting*). Semnul: eroare mică pe antrenare, mare pe test. Cu cât sunt mai multe coloane față de numărul de rânduri, cu atât memorarea e mai ușoară; cu mai multe coloane decât rânduri, regresia poate trece exact prin fiecare punct.

Ca să folosim **toate** cele 79 de coloane, pregătirea se face într-o **linie de procesare** (`Pipeline`): pașii se învață pe antrenare și se aplică identic pe test. Pașii: completăm valorile lipsă (mediana la numere, „Lipsă” la text) și codificăm one-hot coloanele text. Funcția `pregatire()` de mai jos este **recuzita** modulelor 4 și 5: o refolosim neschimbată.

**Prezice.** Antrenăm pe doar 150 de case, cu toate coloanele (după one-hot devin câteva sute). Cât va fi $R^2$ pe antrenare? Dar pe test?

**Cere agentului:**

> Construiește un `Pipeline` care completează lipsurile (mediana pentru numerice, „Lipsă” pentru text), codifică one-hot coloanele text (`handle_unknown="ignore"`) și antrenează o regresie liniară. Antrenează-l pe primele 150, 300, 600 de case din antrenare și pe toate, și compară R² pe antrenare cu R² pe test.

In [ ]:
# Recuzita modulului: pregătirea datelor, ca linie de procesare
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def pregatire(scalare=False):
    """Rețeta de pregătire, învățată DOAR pe antrenare (în Pipeline).
    Numere: mediana în locul valorilor lipsă. Text: „Lipsă” în locul valorilor lipsă, apoi one-hot
    (categoriile noi din test sunt ignorate). La cerere: standardizarea tuturor coloanelor rezultate."""
    text = make_pipeline(SimpleImputer(strategy="constant", fill_value="Lipsă"),
                         OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    coloane = make_column_transformer((SimpleImputer(strategy="median"), make_column_selector(dtype_include="number")),
                                      (text, make_column_selector(dtype_exclude="number")),
                                      verbose_feature_names_out=False)
    return make_pipeline(coloane, StandardScaler()) if scalare else coloane

In [ ]:
# Cod de referință: aceeași regresie, antrenată pe tot mai multe case
rezultate = []
for n in [150, 300, 600, len(X_train)]:
    model = make_pipeline(pregatire(), LinearRegression()).fit(X_train[:n], y_train[:n])
    rezultate.append({"case la antrenare": n,
                      "coloane după one-hot": model[0].transform(X_train[:1]).shape[1],
                      "R2 antrenare": round(model.score(X_train[:n], y_train[:n]), 3),
                      "R2 test": round(model.score(X_test, y_test), 3)})
rezultate = pd.DataFrame(rezultate)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(rezultate["case la antrenare"], rezultate["R2 antrenare"], "o-", label="antrenare")
ax.plot(rezultate["case la antrenare"], rezultate["R2 test"], "o-", label="test")
ax.set_xlabel("case la antrenare")
ax.set_ylabel("R²")
ax.set_title("Puține case: perfect pe antrenare, slab pe test")
ax.legend()
plt.tight_layout()
plt.show()
rezultate

**Verifică.** Pe 150 de case, cu 243 de coloane: $R^2$ 1,000 pe antrenare și 0,381 pe test. Pe toate cele 1.168 de case (301 coloane): 0,937 și 0,887.

**Capcană.** Cu 150 de case și mai multe coloane decât case, modelul trece prin fiecare casă din antrenare ($R^2$ = 1). Pe test e **mai slab decât dreapta cu un singur predictor** din capitolul 2. Un agent care raportează scorul pe antrenare (sau pe toate datele, fără test) îți arată un model „perfect”. Cu mai multe case, diferența dintre antrenare și test se micșorează: mai multe date înseamnă mai puțină memorare. Când nu putem aduce mai multe date, **limităm modelul**: regularizarea din `curs-04b`.

**Pentru client.** „Un model care se potrivește perfect vânzărilor trecute nu e neapărat un model bun. Îl judecăm doar pe case pe care nu le-a văzut.”

---
## Ce urmează

Știm să măsurăm cinstit (împărțire, model de referință, MAE/RMSE/R² pe test) și am văzut cum un model cu prea multă libertate memorează. În `curs-04b` îl ținem în frâu: **regularizarea** (Ridge, Lasso, Elastic Net), alegerea intensității ei prin validare încrucișată și ce îi spunem agenției.

Continuăm cu [curs-04b-regularizarea-regresiei.ipynb](curs-04b-regularizarea-regresiei.ipynb). Puntea curs → seminar este la finalul lui.

---
# Referință

Definiții și detalii pentru acasă. Nu se parcurg la curs.

## R1. Măsuri de eroare pentru regresie

| Măsura | Formula | Unitate | Ce spune |
|---|---|---|---|
| MAE | $\frac{1}{n}\sum_i \lvert y_i - \hat y_i\rvert$ | ca ținta ($) | eroarea „tipică”, ușor de explicat clientului |
| RMSE | $\sqrt{\frac{1}{n}\sum_i (y_i - \hat y_i)^2}$ | ca ținta ($) | pedepsește mai tare greșelile mari; RMSE ≥ MAE |
| MAPE | $\frac{100}{n}\sum_i \left\lvert \frac{y_i - \hat y_i}{y_i}\right\rvert$ | % | eroarea relativă; instabilă când $y$ e aproape de 0 |
| $R^2$ | $1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$ | fără | partea din variație explicată; 1 = perfect, 0 = cât media, negativ = mai rău decât media |

Raportăm măsurile **pe test** și le comparăm cu cele pe antrenare (supraînvățare?) și cu modelul de referință.

## R2. Metoda celor mai mici pătrate

Coeficienții se aleg astfel încât **suma pătratelor reziduurilor** să fie minimă:

$$\text{RSS}(b_0, \dots, b_p) = \sum_{i=1}^{n} \left(y_i - b_0 - b_1 x_{i1} - \dots - b_p x_{ip}\right)^2$$

Matricial, cu $X$ matricea predictorilor (cu o coloană de 1 pentru intercept): $\hat b = (X^\top X)^{-1} X^\top y$. „Pălăria” ($\hat y$, $\hat b$) marchează valori **estimate** din date, spre deosebire de cele reale. Când predictorii sunt foarte corelați sau sunt mai mulți decât rândurile, $X^\top X$ e aproape necalculabil invers: coeficienții devin instabili (capitolele 3 și 4). Regularizarea (`curs-04b`) rezolvă tocmai asta.

## R3. Bias și varianță

Eroarea unui model pe date noi are două surse: **bias** (modelul e prea simplu și greșește sistematic, de exemplu o dreaptă pentru o relație curbă) și **varianță** (modelul e prea flexibil și se schimbă mult de la un eșantion la altul: memorează). Mai multe coloane sau un model mai flexibil scad bias-ul și cresc varianța. Căutăm echilibrul cu cea mai mică eroare pe test. Slide-uri: [curs-04-bias-variance.pdf](curs-04-bias-variance.pdf).

## R4. Ținte asimetrice: logaritmul

Prețurile au o coadă lungă la dreapta (puține case foarte scumpe; asimetria lui `SalePrice` ≈ 1,88). Un model antrenat pe $\log(\text{preț})$ greșește în **procente**, nu în dolari, și e mai puțin tras de casele scumpe. Predicția se întoarce în dolari cu $e^{\hat y}$ (`np.expm1`, dacă s-a folosit `np.log1p`).

## R5. Pentru aprofundare

- [extra-04-regresie-clasica.ipynb](extra-04-regresie-clasica.ipynb): regresia „statistică” cu `statsmodels` (p-values, intervale de încredere, selecție pas cu pas), variabile categoriale cu multe niveluri, valori aberante; pe date despre plămâni, mașini și case (King County). Materialul din anii anteriori.
- Slide-uri: [curs-04-regresie.pdf](curs-04-regresie.pdf), [curs-04-regresori-liniari.pdf](curs-04-regresori-liniari.pdf).

## Jurnal de sesiune

Completează la sfârșit (sau cere agentului: *„închide capitolul”*) și păstrează-l în copia ta a notebook-ului. Dacă agentul rămâne fără tokeni, lipește primerul și acest jurnal unui alt agent și scrie „continuă de aici”.

- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):